In [28]:
# Core libraries
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Resolve project root so we can import from src/
project_root = Path.cwd()
for candidate in [project_root, project_root.parent, project_root / 'indian_housing']:
    if (candidate / 'src').exists() and (candidate / 'data' / 'raw' / 'india_housing_prices.csv').exists():
        project_root = candidate
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load data using the existing ingestion module
from src.data_ingestion import load_data

csv_path = project_root / 'data' / 'raw' / 'india_housing_prices.csv'
df = load_data(csv_path)

# Quick confirmation
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

✅ Data ingestion successful!
Shape: (250000, 23)
Columns: ['ID', 'State', 'City', 'Locality', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status']
Dataset loaded: 250,000 rows × 23 columns


In [29]:
df=df.sample(n=5000,random_state=42)

In [30]:
df.columns=df.columns.str.lower()

In [31]:
df.head(3)

,id,state,city,locality,property_type,bhk,size_in_sqft,price_in_lakhs,price_per_sqft,year_built,furnished_status,floor_no,total_floors,age_of_property,nearby_schools,nearby_hospitals,public_transport_accessibility,parking_space,security,amenities,facing,owner_type,availability_status
38683,38684,Haryana,Gurgaon,Locality_123,Independent House,4,692,256.62,0.37,2022,Semi-furnished,15,24,3,6,6,Medium,Yes,No,"Playground, Pool",South,Builder,Ready_to_Move
64939,64940,Andhra Pradesh,Vishakhapatnam,Locality_74,Apartment,2,3094,86.04,0.03,2015,Furnished,3,18,10,3,8,Medium,No,No,"Gym, Clubhouse, Pool, Garden",South,Builder,Under_Construction
3954,3955,Madhya Pradesh,Bhopal,Locality_486,Apartment,3,4993,237.86,0.05,1995,Unfurnished,19,27,30,10,1,High,No,No,"Gym, Pool, Playground, Clubhouse, Garden",West,Owner,Ready_to_Move


In [32]:
# delet ID and Age of yesr built and price_perSque
df=df.drop(columns=["id","price_per_sqft","year_built"])

In [33]:
X=df.drop(["price_in_lakhs"],axis=1)
y=df["price_in_lakhs"]

In [34]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

In [35]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((4000, 19), (1000, 19), (4000,), (1000,))

In [ ]:
5. Implement phase0 as per the @docs/Implementation-plan.md   # Load the Phase-3 preprocessor that was built and saved by `src.preprocessing`
from joblib import load

# The saved preprocessor includes amenity extraction, target-encoding, one-hot and scaling
preprocessor_saved = load("models/preprocessor.joblib")
feature_names = preprocessor_saved.get_feature_names_out()

# Apply to train/test splits
X_train_encoded = preprocessor_saved.transform(X_train)
X_test_encoded = preprocessor_saved.transform(X_test)

print("Preprocessor loaded; transformed shapes:", X_train_encoded.shape, X_test_encoded.shape)


In [ ]:
# This cell was replaced: preprocessing is applied in the earlier cell (preprocessor_saved.transform).
# No action required here.


c:\Users\a\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [ ]:
# The saved preprocessor already scales numeric features. Use the transformed arrays directly.
X_train_scaled = X_train_encoded
X_test_scaled = X_test_encoded


In [39]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()

model.fit(X_train_encoded,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


1. Baseline Models

In [40]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Baseline Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
y_pred_lr = lin_reg.predict(X_test_scaled)

# Baseline Random Forest
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

# Evaluation function
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} → MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")

evaluate_model(y_test, y_pred_lr, "Linear Regression")
evaluate_model(y_test, y_pred_rf, "Random Forest")


Linear Regression → MAE: 132.9676, RMSE: 159.6145, R²: -0.2234
Random Forest → MAE: 126.4954, RMSE: 146.3401, R²: -0.0284


2. Advanced Models (XGBoost & LightGBM)

In [41]:
import xgboost as xgb
import lightgbm as lgb

# XGBoost
xgb_model = xgb.XGBRegressor(random_state=42)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)
evaluate_model(y_test, y_pred_xgb, "XGBoost")

# LightGBM
lgb_model = lgb.LGBMRegressor(random_state=42)
lgb_model.fit(X_train_scaled, y_train)
y_pred_lgb = lgb_model.predict(X_test_scaled)
evaluate_model(y_test, y_pred_lgb, "LightGBM")


XGBoost → MAE: 127.4350, RMSE: 149.8789, R²: -0.0787
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009919 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 685
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 108
[LightGBM] [Info] Start training from score 251.973920
LightGBM → MAE: 127.5526, RMSE: 148.2756, R²: -0.0558


3. Hyperparameter Tuning (Optuna Example)

In [42]:
import optuna

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 3, 15)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)

    model = xgb.XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    return mean_squared_error(y_test, preds)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

best_params = study.best_params
print("Best Hyperparameters:", best_params)

best_model = xgb.XGBRegressor(**best_params, random_state=42)
best_model.fit(X_train_scaled, y_train)


[I 2026-08-09 12:50:01,284] A new study created in memory with name: no-name-e8c4ee8a-15f8-4b6d-a961-c76b2209fdca
[I 2026-08-09 12:50:07,069] Trial 0 finished with value: 23341.104450751587 and parameters: {'n_estimators': 308, 'max_depth': 10, 'learning_rate': 0.11765752218423504}. Best is trial 0 with value: 23341.104450751587.
[I 2026-08-09 12:50:16,512] Trial 1 finished with value: 24784.78608535158 and parameters: {'n_estimators': 409, 'max_depth': 11, 'learning_rate': 0.16574535743222119}. Best is trial 0 with value: 23341.104450751587.
[I 2026-08-09 12:50:18,571] Trial 2 finished with value: 20822.883647933573 and parameters: {'n_estimators': 203, 'max_depth': 4, 'learning_rate': 0.03893121596533169}. Best is trial 2 with value: 20822.883647933573.
[I 2026-08-09 12:50:24,998] Trial 3 finished with value: 24967.50079318784 and parameters: {'n_estimators': 176, 'max_depth': 15, 'learning_rate': 0.22101451056960555}. Best is trial 2 with value: 20822.883647933573.
[I 2026-08-09 12:

Best Hyperparameters: {'n_estimators': 335, 'max_depth': 3, 'learning_rate': 0.01065765019182738}


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
import shap
import matplotlib.pyplot as plt

# Use shap.Explainer with the fast tree algorithm when available
# Try Tree SHAP first (fast). If it fails (XGBoost version mismatch), fallback to
# sklearn permutation importances which are slower but robust.
try:
    explainer = shap.Explainer(best_model, algorithm="tree")

    # Build a DataFrame for feature names (after scaling)
    feature_names = preprocessor.get_feature_names_out()
    X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

    # Compute SHAP values on a small subset to keep runtime reasonable
    sample_n = min(200, X_test_scaled.shape[0])
    shap_values = explainer(X_test_scaled[:sample_n])
    shap.plots.summary(shap_values, features=X_test_df.iloc[:sample_n])
except Exception as e:
    print("Tree SHAP failed, falling back to permutation importance:", e)
    from sklearn.inspection import permutation_importance

    feature_names = preprocessor.get_feature_names_out()
    sample_n = min(500, X_test_scaled.shape[0])
    r = permutation_importance(best_model, X_test_scaled[:sample_n], y_test.iloc[:sample_n], n_repeats=10, random_state=42, n_jobs=-1)
    importances = pd.Series(r.importances_mean, index=feature_names).sort_values(ascending=False)
    importances.head(20).plot(kind='barh')
    plt.gca().invert_yaxis()
    plt.title("Permutation feature importances (fallback)")


Tree SHAP failed, falling back to permutation importance: could not convert string to float: '[2.5197392E2]'


5. Save Final Model

In [47]:
import joblib
joblib.dump(best_model, "best_model.pkl")
print("Final model saved as best_model.pkl")


Final model saved as best_model.pkl
